# Autograd 与反向传播

## 学习目标

能够追踪标量损失的计算图，解释叶子张量、梯度累积、`detach` 和禁用梯度。


## 概念模型与执行路径

前向运算构建动态图，反向传播沿图应用链式法则。梯度只自动累积到需要梯度的叶子张量；优化器更新前必须清理旧梯度。


### 实验 1


In [1]:
import torch
x = torch.tensor([1.0, 2.0, 3.0])
w = torch.tensor([0.5, 0.5, 0.5], requires_grad=True)
prediction = (x * w).sum()
loss = (prediction - 4.0) ** 2
print("prediction:", prediction.item(), "loss:", loss.item(), "grad_fn:", loss.grad_fn)
loss.backward()
print("d(loss)/d(w):", w.grad)


prediction: 3.0 loss: 1.0 grad_fn: <PowBackward0 object at 0x10c5a55d0>
d(loss)/d(w): tensor([-2., -4., -6.])


### 实验 2


In [2]:
first_gradient = w.grad.clone()
((x * w).sum() - 4.0).pow(2).backward()
print("gradient accumulated:", w.grad)
torch.testing.assert_close(w.grad, first_gradient * 2)
w.grad = None


gradient accumulated: tensor([ -4.,  -8., -12.])


### 实验 3


In [3]:
with torch.no_grad():
    inference_value = (x * w).sum()
detached = prediction.detach()
print("no_grad requires_grad:", inference_value.requires_grad)
print("detached requires_grad:", detached.requires_grad)


no_grad requires_grad: False
detached requires_grad: False


### 实验 4


In [4]:
w2 = torch.tensor([0.5, 0.5, 0.5], requires_grad=True)
loss2 = ((x * w2).sum() - 4.0) ** 2
analytical = torch.autograd.grad(loss2, w2)[0]
print("autograd.grad:", analytical)


autograd.grad: tensor([-2., -4., -6.])


## 底层机制

PyTorch 保存反向传播所需的中间值，因此带计算图的张量长期保存在列表中会占用内存。`backward()` 默认释放图；需要二阶梯度时才使用 `create_graph=True`。


## 检查点

解释为什么第二次反向传播重新执行了前向表达式，而不能直接对同一个 `loss` 再调用 `backward()`。


## 试一试

把损失改为绝对误差，观察梯度如何变化；再用有限差分近似验证平方误差梯度。


## 常见错误与调试

忘记清梯度、对非标量输出直接 backward、原地修改反向所需张量、在训练阶段误用 `no_grad`。
